In [ ]:
# Настройка окружения
!pip install datasets==3.2.0 open_clip_torch==2.29.0 nltk==3.9.1

In [ ]:
import math
import time
from typing import Tuple, List, Dict

import matplotlib.pyplot as plt
import nltk
import torch
from datasets import load_dataset
from nltk.corpus import wordnet
from open_clip import tokenizer
from PIL import Image

import open_clip
from sklearn.metrics import accuracy_score

# > Классификация. CLIP

## Описание задачи

В этом задании мы будем решать задачу классификации изображений, используя архитектуру CLIP (Contrastive Language–Image Pre-training). В качестве примера мы воспользуемся уже знакомым нам набором данных Tiny ImageNet. Однако на этот раз мы не будем обучать модель с нуля. Вместо этого мы применим Zero Shot —  классификацию, используя предобученную модель CLIP.

## План
1. **Подготовка модели**: загрузим предобученную модель CLIP из библиотеки open_clip_torch.
2. **Подготовка данных**: подготовим изображения и текстовые метки (классы) для инференса модели.
3. **Подготовка пайплайна**: создадим пайплайн для инференса и оценки качества модели на заданном наборе данных.
4. **Сравнение моделей**: сравним между собой предобученные модели.

# > Подготовка модели

Как уже обсуждали ранее на практических занятиях, реализацию CLIP мы возьмём из репозитория [open_clip](https://github.com/mlfoundations/open_clip).

Ещё раз посмотрим список доступных моделей и для примера возьмём первую из них.

In [ ]:
pretrain_models = open_clip.list_pretrained()
print(pretrain_models)

Реализуем функцию `load_model`, которая инициализирует и подготавливает модель для инференса. Для примера будем рассматривать модель ('RN50', 'openai').

In [ ]:
def load_model(model_name, pretrained) -> open_clip.model.CLIP:
    """
    Загружает и подготавливает модель CLIP для использования.

    :param model_name: Имя модели, которую необходимо загрузить.
    :param pretrained: Датасет, на котором обучалась модель.

    :return model, preprocess: Кортеж, содержащий загруженную модель CLIP и объект обработки изображений.
    """
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, preprocess

model, preprocess = load_model(model_name='RN50', pretrained='openai')

Внесите в LMS ответ на следующий вопрос:

Задание 1. Сколько уникальных архитектур моделей доступно в `open_clip`?

In [ ]:
num_models = # Ваш код здесь
print(num_models)

# > Подготовка данных

Скачаем напрямую с `huggingface` датасет `tiny-imagenet`. Так как мы не планируем обучать модель, а хотим оценить её качество, то возьмём только валидационную часть датасета.

In [ ]:
dataset = load_dataset("zh-plus/tiny-imagenet", split="valid")

Давайте ещё раз взглянем на датасет и разберёмся, в каком формате представлены наши данные и как их можно извлекать.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 3))
for plot_index, data_index in enumerate(range(0, 1000, 100)):
  image = dataset["image"][data_index]
  label = dataset["label"][data_index]
  axes[plot_index].imshow(image)
  axes[plot_index].set_title(f"Class: {label}")
  axes[plot_index].axis('off')


Изучите датасет и внесите в LMS ответы на следующие вопросы:

- **Задание 2.1.** Cколько всего примеров в датасете?
- **Задание 2.2.** Cколько различных классов в датасете?
- **Задание 2.3.** Cколько примеров каждого класса в датасете?

## Подготовка текстовых описаний классов

Мы посмотрели, как выглядят исходные данные, теперь научимся их подготавливать для работы с моделями CLIP. Как вы могли догадаться, номера классов не подойдут нам для описания изображений. Нам нужно преобразовать их в читаемый текст.
Посмотрим, есть ли в нашем датасете текстовое описание классов.

In [ ]:
names = dataset.features['label'].names
print(names)

В таком виде имена классов тоже не подойдут, поскольку для использования модели CLIP нам нужно задать текстовый запрос, с которым модель будет сравнивать изображения.

На самом деле эти странные имена — `'n01443537'` — являются кодировкой классов из датасета `ImageNet` в «базе данных» `WordNet` и их можно сопоставить с текстовым названием класса. Для этого можно воспользоваться библиотекой `nltk`, которая умеет работать с такой кодировкой.


In [ ]:
nltk.download('wordnet')


Не вдаваясь в подробности: чтобы получить имя класса, можно воспользоваться следующим кодом:

In [ ]:
name = names[0]
synset = wordnet.synset_from_pos_and_offset('n', int(name[1:]))
text = synset.lemma_names()[0]

print(f'{name} -> {text}')

Давайте для удобства работы с датасетом создадим функцию, которая будет сопоставлять индекс класса (из `dataset["label"]`) с его названием.

Индексы классов совпадают с их индексом в списке `dataset.features['label'].names`.

**Задание 3.** Реализуйте функцию `label_index_to_name`, которая будет сопоставлять индексы классов с их названием. На вход функция принимает индекс класса, на выход выдаёт его имя. Если передан некорректный индекс, необходимо вернуть ошибку `ValueError` c описанием `f"Invalid index: {index}"`.


In [ ]:
def label_index_to_name(index: int, names) -> str:
  """
  Преобразует индекс метки в человекочитаемое имя, используя заданный набор имён.

  :param index: Целочисленный индекс метки из списка имён.
  :param names: Список имён в кодировке WordNet, соответствующих меткам в наборе данных.
  :return: Человекочитаемое имя класса, соответствующее заданному индексу метки.

  :raises ValueError: Если переданный индекс не является целым числом или находится вне диапазона допустимых значений.
  """
  # Ваш код здесь
  return name


Пример использования:

In [ ]:
image_index = 1000
labels = dataset["label"]
label_index = labels[image_index]
print(label_index)

names = dataset.features['label'].names
print(names)

name = label_index_to_name(index=label_index, names=names)
print(name)

## Создание текстовых описаний

Создадим список человекочитаемых имён классов и будем работать уже с ним.

In [ ]:
labels_names = [label_index_to_name(i, names) for i in range(len(names))]
print(labels_names)

**Задание 4.** Теперь, когда у вас есть список классов, реализуйте функцию create_prompts, которая будет формировать для наших классов текстовое описание `(prompt)`. На вход она будет принимать список имён классов (человекочитаемых) и шаблон, в который они будут подставлены для формирования текстового описания.

In [ ]:
def create_prompts(labels_names: list[str], template: str) -> list[str]:
  """
  Создаёт список текстовых подсказок (prompts) на основе заданного шаблона и списка имён меток.

  :param labels_names: Список строк, содержащий имена меток, которые будут подставлены в шаблон.
  :param template: Строка-шаблон, в которую подставляются имена меток из списка; должна содержать один маркер подстановки "{}".
  :return: Список строк, где каждая строка представляет собой шаблон с подставленным именем метки.
  """
  # Ваш код здесь
  return prompts

Создадим список промптов для нашего датасета.

In [ ]:
template = "a photo of {}"
prompts = create_prompts(labels_names, template)
print(prompts)

# > Подготовка пайплайна
### Энкодинг текстов

Теперь, когда у нас есть читаемые описания классов, нам нужно извлечь из них эмбеддинги с помощью энкодера модели CLIP. Напишите функцию для преобразования наших запросов в эмбеддинг. Для этого необходимо токенизировать текст при помощи `tokenizer`, прогнать его через энкодер модели.

**Задание 5.** Реализуйте на основе материалов практического занятия функцию `get_text_embeddings`.

In [ ]:
def get_text_embeddings(model, prompt: list[str]) -> torch.Tensor:
    """
    Получает текстовые эмбеддинги для списка запросов.

    :param model: Модель для инференса.
    :param prompt: Список строк, каждая из которых представляет собой текстовый запрос.
    :return: Тензор PyTorch, содержащий нормализованные эмбеддинги текстов.
    """
    # Ваш код здесь
    return text_embeddings

Пример использования:

In [ ]:
text_embeddings = get_text_embeddings(model, prompts)
print(text_embeddings.shape)

## Энкодинг изображений

Теперь нам необходимо подготовить изображения для инференса модели. Для этого воспользуемся функцией `preprocess`, которую мы получили при инициализации модели, и получим эмбеддинг при помощи энкодера.

**Задание 6.** Реализуйте функцию `get_image_embedding`.



In [ ]:
def get_image_embedding(model, image: torch.Tensor) -> torch.Tensor:
  """
  Получает эмбеддинг изображения с использованием предварительно обученной модели.

  :param model: Модель для инференса.
  :param image: Тензор PyTorch, представляющий изображение, для которого необходимо получить эмбеддинг.
  :return: Тензор PyTorch, содержащий эмбеддинг изображения.
  """
  # Ваш код здесь
  return image_embedding


Пример использования:

In [ ]:
EXAMPLE_IMAGE_INDEX = 1
image = dataset["image"][EXAMPLE_IMAGE_INDEX]
img_tensor = preprocess(image).unsqueeze(0).float().to('cuda')
print(img_tensor.shape)

image_embedding = get_image_embedding(model, img_tensor)
print(image_embedding.shape)

### Инференс модели
**Задание 7.** На предыдущих шагах мы получили эмбеддинги для текстов и изображения. Реализуйте функцию `get_probs`, которая на вход получает эмбеддинги текстов для каждого класса и эмбеддинг изображения. Функция должна возвращать вероятности принадлежности изображения к каждому из классов. Вероятности должны быть нормированы так, чтобы их сумма равнялась 1.

*В качестве множителя, на который будет помножено матричное перемножение векторов, возьмите 100. Это применяют, чтобы увеличить разницу в значениях, что делает softmax-распределение более «острым», то есть наибольший элемент будет иметь более высокую «уверенность».*

In [ ]:
def get_probs(image_embedding, text_embeddings):
  """
  Вычисляет вероятности соответствия изображения текстовым подсказкам на основе эмбеддингов.

  :param image_embedding: Тензор PyTorch, представляющий эмбеддинг изображения.
  :param text_embeddings: Тензор PyTorch, содержащий эмбеддинги текстовых подсказок.
  :return: Тензор PyTorch, представляющий вероятности соответствия между изображением и текстовыми подсказками.
  """
  # Ваш код здесь
  return probs

probs = get_probs(image_embedding, text_embeddings)



### Оценка качества классификации

**Задание 8.** Для оценки качества классификации реализуйте topK_accuracy.

In [ ]:
def calculate_topk_accuracy(probs, gts, k=1):
    """
    Вычисляет top-k accuracy.

    :param probs: Тензор вероятностей размера (num_samples, num_classes).
    :param gts: Истинные метки классов.
    :param k: Параметр top-k, по умолчанию равен 1.
    :return: Точность классификации top-k.
    """
    # Ваш код здесь
    return accuracy

Пример использования:

In [ ]:
probs = torch.tensor([
        [0.2, 0.5, 0.3],
        [0.1, 0.4, 0.5],
        [0.7, 0.2, 0.1]
])
gts = [1, 2, 0]
k = 1
accuracy = calculate_topk_accuracy(probs, gts, k)

print(accuracy)

1.0


# > Сравнение моделей
Мы реализовали весь необходимый функционал для инференса модели и оценки качества классификации. Теперь нам предстоит протестировать некоторые из предобученных моделей и сравнить их между собой.

Хорошей практикой в данном случае будет объединить наши функции в некий пайплайн, который мы будем запускать.

Ниже я предложу один из вариантов того, как это может выглядеть. Как именно он будет реализован в данном случае, не важно, главное — чтобы для каждой модели вы могли получить значения Top-1/5/10 Accuracy на нашем датасете.

In [ ]:
class CLIPClassifier:
    def __init__(self, model_name, pretrain):
        """Инициализация предобученной модели из open_clip"""
        # Ваш код здесь
        pass

    def create_prompts(self, labels_names: List[str], template: str) -> List[str]:
        """Создание запросов в модель на основе имён классов и шаблона"""
        # Ваш код здесь
        pass

    def get_text_embeddings(self, prompt: List[str]) -> torch.Tensor:
        """Получение эмбеддингов для запросов в модель"""
        # Ваш код здесь
        pass

    def get_image_embedding(self, image: List[str]) -> torch.Tensor:
        """Получение эмбеддинга для изображения"""
        # Ваш код здесь
        pass

    def classify(self, images, class_labels, template="a photo of a {}"):
        """Классификация изображений на основе эмбеддингов текста и изображений"""
        # Ваш код здесь
        pass

    def calculate_topk_accuracy(self, probs, gts, k=1):
        """
        Вычисляет top-k accuracy.

        :param probs: Тензор вероятностей размера (num_samples, num_classes).
        :param gts: Истинные метки классов.
        :param k: Параметр top-k, по умолчанию равен 1.
        :return: Точность классификации top-k.
        """
        pass

# Пример использования
templates = ['a photo of {}']
for name, pretrain in [('RN50', 'openai'), ('RN50', 'yfcc15m'), ('RN50', 'cc12m')]:
  print(f"Model: {name}")
  print(f"Pretrain: {pretrain}")
  for template in templates:
    print(f"Template: {template}")
    classifier = CLIPClassifier(name, pretrain)
    images = dataset["image"]
    probs = classifier.classify(images, labels_names, template)
    gts = dataset["label"]

    top1_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=1)
    print(f"Top-1 Accuracy: {top1_accuracy:.2f}")

    top5_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=5)
    print(f"Top-5 Accuracy: {top5_accuracy:.2f}")

    top10_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=10)
    print(f"Top-10 Accuracy: {top10_accuracy:.2f}"

**Задание 9.1.** Модель, предобученная на каком из датасетов показывает лучшее значение Top-1 Accuracy?

Шаблон для запросов: "a photo of {}".

In [ ]:
templates = ['a photo of {}']
pretrain_models = open_clip.list_pretrained()
for name, pretrain in [('RN50', 'openai'), ('RN50', 'yfcc15m'), ('RN50', 'cc12m')]:
  print(f"Model: {name}")
  print(f"Pretrain: {pretrain}")
  for template in templates:
    print(f"Template: {template}")
    classifier = CLIPClassifier(name, pretrain)

    images = dataset["image"]

    probs = classifier.classify(images, labels_names, template)
    gts = dataset["label"]

    top1_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=1)
    print(f"Top-1 Accuracy: {top1_accuracy:.2f}")

    top5_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=5)
    print(f"Top-5 Accuracy: {top5_accuracy:.2f}")

    top10_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=10)
    print(f"Top-10 Accuracy: {top10_accuracy:.2f}")
    print()

**Задание 9.2.** Какой из шаблонов позволяет добиться наилучшего качества Top-5 Accuracy для модели ViT-B-32, laion2b_s34b_b79k?

In [ ]:
templates = ['{}', 'a photo of {}', 'photo of {} from ImageNet dataset', 'a photo showing a {}']
pretrain_models = open_clip.list_pretrained()

name = 'ViT-B-32'
pretrain = 'laion2b_s34b_b79k'
print(f"Model: {name}")
print(f"Pretrain: {pretrain}")
for template in templates:
  print(f"Template: {template}")
  classifier = CLIPClassifier(name, pretrain)

  images = dataset["image"]


  probs = classifier.classify(images, labels_names, template)
  gts = dataset["label"]

  top1_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=1)
  print(f"Top-1 Accuracy: {top1_accuracy:.2f}")

  top5_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=5)
  print(f"Top-5 Accuracy: {top5_accuracy:.2f}")

  top10_accuracy = classifier.calculate_topk_accuracy(probs, gts, k=10)
  print(f"Top-10 Accuracy: {top10_accuracy:.2f}")
  print()